# End-to-End Log Detection & Segmentation Pipeline

This document describes the complete two-stage system used to detect logs in video, stitch them into continuous images, segment the main log body, detect bark regions, and generate a fully annotated final output.

---

# 1. System Overview

The pipeline consists of **two major stages**, each running on different hardware:

### **Stage A — Jetson (Video → Stitched Logs)**
- Detects when a log is present in each video frame.
- Uses lightweight feature extraction + PCA + SVM.
- Groups consecutive “log frames” into intervals.
- Horizontally stitches frames from each interval into a final log image.

### **Stage B — GPU (Stitched Log → Log Boundary + Bark Segmentation)**
- Splits log into halves with overlap.
- Segments the log boundary using YOLO.
- Normalizes crop heights (align left/right halves).
- Performs bark segmentation.
- Restitches and merges segmentations into one unified result.

---

# 2. Stage A — Log Detection From Video (Jetson)

## 2.1 Region of Interest (ROI)
- Only the region where the log appears is processed.
- Eliminates background noise and drastically reduces computation.

## 2.2 Frame Differencing
To isolate motion:

- For each frame t, compute:  
  **diff = ROI(t) – ROI(t-1)**  
- Highlights objects that move (logs).  
- Background is canceled out.  
- This difference image becomes the input to EfficientNet.

![My Image](frames.png)

**Why this works:**  
Logs move predictably across the belt; the background does not. Differencing gives strong motion features even under poor lighting.

## 2.3 EfficientNet Feature Extraction
- EfficientNet (pretrained on ImageNet) is used **only as a frozen feature extractor**.
- Intermediate layer activations are collected.
- A single layer’s features (e.g., layer 7) are flattened into a vector.

## 2.4 Standardization + PCA (Dimensionality Reduction)
- Features are passed through **StandardScaler**.
- PCA reduces thousands of feature dimensions to **4 principal components**.

This produces a compact, stable representation suitable for SVM classification.

![My Image](fi.png)

## 2.5 SVM Classification (Log vs No-Log)
- RBF kernel SVM.
- Hyperparameter tuning using GridSearchCV.
- Objective metric: **recall** (to avoid missing any log frames).
- Produces sequence of predictions:  
  0 0 1 1 1 1 0 0 …

A decision score plot looks like:

![My Image](decision.png)

## 2.6 Log Interval Detection & Frame Stitching
- Consecutive “1” predictions form **log intervals**.
- Frames within each interval are stitched horizontally.
- Resulting image represents the entire passing log surface.

Output example:  
{job_id}_stitched.jpg
![My Image](log8_stitched.jpg)
![My Image](log1_stitched.jpg)
![My Image](log11_stitched.jpg)
![My Image](log12_stitched.jpg)

---

# 3. Stage B — Log + Bark Segmentation (GPU)

Two YOLO segmentation models are used:

### **Log Segmentation Model**
- Single class: log
- Predicts the outer log boundary polygon

### **Bark Segmentation Model**
- Multi-instance segmentation
- Predicts bark surface fragments

Both models were trained with polygon masks and follow similar training schedules.

---

# 4. Log Segmentation Pipeline (Stage B — Part 1 & 2)
![My Image](logannot.png)
## 4.1 Half-Split With 3% Horizontal Overlap
- Stitched log is split into left/right halves.
- A **3% overlap** ensures the log boundary is not cut at the split.

**Rationale:**  
YOLO segmentation accuracy drops at image edges; the overlap prevents boundary discontinuity.

## 4.2 YOLO Log Segmentation
- Each half is fed into the YOLO log model.
- Outputs polygon masks representing the log outline.

### **Model Training Notes**
YOLO training followed a **3-phase schedule**:

1. **Phase 1 — Freeze Backbone**
   - Train only mask heads & upper layers.
   - Stabilizes early learning of segmentation masks.

2. **Phase 2 — Unfreeze All Layers**
   - Full network trains jointly.
   - Strong augmentations: flips, scale jitter, color jitter.

3. **Phase 3 — Fine-Tune**
   - Lower learning rate.
   - Reduced augmentation.
   - Optimizes mask smoothness and contour consistency.

![My Image](log1_stitched_left.jpg)
![My Image](log1_stitched_right.jpg)
## 4.3 Greenify + Unified Height Cropping
To prepare for bark segmentation:

- Convert YOLO polygons to a binary mask.
- Morphologically smooth the result.
- Find bounding rectangle of the log.
- Compute shared **top and bottom** across both halves.
- Crop both halves to **identical vertical height**.
- Fill non-log areas with a uniform **light green** background.
- Save cropping offsets for later polygon alignment.

---

# 5. Bark Segmentation Pipeline (Stage B — Part 3)
![My Image](barkannot.png)
## 5.1 Split Into Two Quarters (No Overlap)
- Each height-normalized half is split again into left/right quarters.
- This increases YOLO’s effective resolution for bark texture detection.

## 5.2 YOLO Bark Segmentation
- Bark model predicts multiple bark polygons per quarter.
- White outlines are drawn onto the quarter images.

### **Training Notes**
- YOLOv5 segmentation model.
- Polygon masks for bark regions.
- High texture variability → heavier augmentation:
  - color shifts
  - random crops
  - noise injection
- Allows model to generalize to different species and lighting.

## 5.3 Restitch Quarters Back Into Halves
- Left and right bark quarters are stitched horizontally.
- Produces two bark-annotated halves (left + right).

---

# 6. Final Log Fusion Pipeline (Stage B — Part 4)

## 6.1 Remove Overlap & Restitch Final Bark Image
- The 3% overlap is removed from the left bark half.
- Right bark half is placed flush against the cropped left half.
- Right half takes precedence (avoids double seams).

## 6.2 Reconstruct the Full Log Boundary
Using YOLO log polygons + saved offsets:

1. Load original log polygons (from left and right halves).  
2. Undo the cropping offsets from greenify stage.  
3. Apply horizontal shift for right-side polygons.  
4. Convert polygons to Shapely geometries.  
5. Compute geometric **union** to form a single continuous boundary.  
6. Rasterize, morphologically smooth, and draw the final outline in **red**.

## 6.3 Final Output
A complete log surface visualization containing:

- Bark outlines (white)
- Unified log boundary (red)
- Fully aligned stitched log body  
- No seams or polygon discontinuities

Result example:  

![My Image](log2_stitched_final.jpg)
![My Image](log5_stitched_final.jpg) 
![My Image](log8_stitched_final.jpg)
![My Image](log28_stitched_final.jpg)
![My Image](log29_stitched_final.jpg)
![My Image](log32_stitched_final.jpg)
---

# 7. Summary of Key Concepts

- **Frame differencing** amplifies motion and removes background reliably.
- **EfficientNet (frozen)** is a high-quality universal feature extractor.
- **StandardScaler + PCA (4D)** produces a compact, stable representation.
- **SVM** performs accurate binary frame classification on lightweight hardware.
- **YOLO segmentation** handles complex geometry for log boundary & bark.
- **Overlap logic** prevents boundary breaks across halves.
- **Unified vertical cropping** ensures geometrically consistent segmentation.
- **Shapely union** merges polygons into one final contour.
- **Pipeline modularity** supports multi-hardware operation (Jetson + GPU).

